In [39]:
# Lees de opgeschoonde dataset; laad jaar en pagina-aantal als gehele getallen.
from pathlib import Path

import pandas as pd
import numpy as np

dataset_path = Path("datasets") / "processed" / "catalogus_opgeschoond.csv"
df = pd.read_csv(
    dataset_path,
    encoding="utf-8",
    sep=";",
    dtype={"aantal_paginas": "Int64", "publicatiejaar": "Int64"},
)
titles = df["Titel"].tolist()
# Toon alle kolomnamen van de opgeschoonde dataset.
df.columns.tolist()

['Titel',
 'Auteur',
 'Korte omschrijving',
 'Type materiaal',
 'Niveau',
 'genre',
 'trefwoorden',
 'aantal_paginas',
 'doelgroep',
 'publicatiejaar',
 'beschikbaar_formaat',
 'cataloguscode',
 'Gemiddelde beoordeling']

In [40]:
# De tokenizers staan in functions.py, zodat recommender.py en de notebooks dezelfde gebruiken.
# Zie dat bestand voor de uitleg per functie; de uitwerking stap voor stap staat in 03 bow_tfidf.ipynb.
from functions import (clean, tokenize, tokenize_stop, tokenize_nostop,
                       tokenize_stemmed, tokenize_stemmed_nostop,
                       stop_nl, CUSTOM_STOPWORDS, all_stop_words)


In [41]:
from sklearn.feature_extraction.text import CountVectorizer

short_descriptions = df["Korte omschrijving"].tolist()
# We geven de basis-tokenizer mee; die wordt later uitgebreid (stopwoorden, stemming).
cv = CountVectorizer(tokenizer=tokenize, token_pattern=None, lowercase=False)
X_books = cv.fit_transform(short_descriptions)

print("Vocabulary:", cv.get_feature_names_out())
print("BoW Matrix:\n", X_books.toarray())

print("vocabulaire:", list(cv.get_feature_names_out()))
pd.DataFrame(X_books.toarray(), columns=cv.get_feature_names_out(), index=df["Titel"]).head(10)

Vocabulary: ['aan' 'aandacht' 'aangrijpend' 'aangrijpende' 'aanleiding'
 'aantrekkingskracht' 'aanwijzingen' 'academische' 'acceptatie'
 'achtergrond' 'achtergronden' 'achtergrondinformatie' 'achterhalen'
 'achtervolging' 'actualiteit' 'actueel' 'actuele' 'administratie'
 'advocaat' 'af' 'afgesloten' 'afhankelijkheid' 'afscheid' 'afspeelt'
 'afspraak' 'agenten' 'ageverhaal' 'ai' 'alledaagse' 'alles' 'als'
 'ambitie' 'amsterdam' 'analyseert' 'anders' 'angst' 'anne' 'arbeidsmarkt'
 'artikelen' 'auteur' 'authenticiteit' 'avontuur' 'balans' 'band'
 'bedreigende' 'bedreiging' 'bedrog' 'beeld' 'beeldvorming' 'begin'
 'beginnende' 'begrijpelijk' 'begrijpelijke' 'behoefte' 'bekende'
 'bekenden' 'bekendste' 'bekentenis' 'belangrijk' 'belangrijke' 'belicht'
 'benauwende' 'bepalen' 'beproefd' 'beroemde' 'beschaving' 'beschermen'
 'beschouwingen' 'beschreven' 'beschrijft' 'beslissing' 'beslissingen'
 'bestaan' 'bestemmingen' 'betekenis' 'betekenisvol' 'beter' 'betere'
 'betrokken' 'betrokkenen' 'b

,aan,aandacht,aangrijpend,aangrijpende,aanleiding,aantrekkingskracht,aanwijzingen,academische,acceptatie,achtergrond,...,zoeken,zoekt,zoektocht,zomerse,zonder,zorg,zorgen,zwaar,zwangerschap,én
Titel,,,,,,,,,,,,,,,,,,,,,
het dagboek van anne frank,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
blauw water,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
boer vindt vrouw,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
cadeau voor u!,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
de biecht,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
de familie blues,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
de favoriete plekken van...,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
de man van de bloemen,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
de ontmoeting,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [42]:
print("vorm van de matrix (documenten x woorden):", X_books.shape)
print("opgeslagen getallen (niet-nullen):", X_books.nnz, "van", X_books.shape[0] * X_books.shape[1])

# Hoe vaak komt elk woord voor, over alle documenten samen?
# X_books.sum(axis=0) telt per kolom (= per woord) alle rijen (= documenten) bij elkaar op.
# Het resultaat is een matrix met één rij, bijvoorbeeld [[63 60 53 ...]]. Met .A1 maak je daar een
# gewone, platte rij getallen van (zonder die extra "rij"), zodat pandas er een Series van kan maken.
# De index van die Series zijn de woorden, in dezelfde volgorde als de kolommen.
freq = pd.Series(X_books.sum(axis=0).A1, index=cv.get_feature_names_out())
freq.sort_values(ascending=False).head(25)

vorm van de matrix (documenten x woorden): (283, 881)
opgeslagen getallen (niet-nullen): 3981 van 249323


en             247
een            214
de             131
over           123
het             85
van             74
verhaal         69
met             61
waarin          57
in              45
keuzes          42
te              35
die             35
op              31
naar            31
voor            31
liefde          30
staan           29
centraal        29
roman           29
vriendschap     26
maken           25
identiteit      25
geheimen        25
leven           24
dtype: int64

In [43]:
# De stopwoordenlijst (NLTK Nederlands) komt mee uit functions.py; hier alleen even kijken wat erin zit.
print("NLTK Nederlandse stopwoorden:", len(stop_nl))
print(sorted(stop_nl)[:40], "...")
print("eigen aanvulling:", CUSTOM_STOPWORDS)


NLTK Nederlandse stopwoorden: 101
['aan', 'al', 'alles', 'als', 'altijd', 'andere', 'ben', 'bij', 'daar', 'dan', 'dat', 'de', 'der', 'deze', 'die', 'dit', 'doch', 'doen', 'door', 'dus', 'een', 'eens', 'en', 'er', 'ge', 'geen', 'geweest', 'haar', 'had', 'heb', 'hebben', 'heeft', 'hem', 'het', 'hier', 'hij', 'hoe', 'hun', 'iemand', 'iets'] ...
eigen aanvulling: {'meisje', 'verhaal', 'jaar', 'boek', 'jongen'}


In [44]:
# Eerste uitbreiding van de tokenizer: de standaard stopwoorden weglaten (tokenize_stop uit functions.py).
cv_stop = CountVectorizer(tokenizer=tokenize_stop, token_pattern=None, lowercase=False)
X_stop = cv_stop.fit_transform(short_descriptions)
print("vocabulaire zonder stopwoorden:", X_stop.shape[1], "woorden (was", X_books.shape[1], ")")

# Woordfrequenties, net als hierboven (.A1: zie de uitleg bij `freq`).
freq_stop = pd.Series(X_stop.sum(axis=0).A1, index=cv_stop.get_feature_names_out())
freq_stop.sort_values(ascending=False).head(15)

vocabulaire zonder stopwoorden: 824 woorden (was 881 )


verhaal          69
waarin           57
keuzes           42
liefde           30
centraal         29
staan            29
roman            29
vriendschap      26
identiteit       25
maken            25
geheimen         25
leven            24
jongeren         23
persoonlijke     22
toegankelijke    20
dtype: int64

TF-IDF

In [45]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Kies een tokenizer uit functions.py: tokenize_stop (NLTK-lijst), tokenize_nostop (plus eigen aanvulling),
# tokenize_stemmed_nostop (plus stemming). Noteer de keuze in het keuzelog.
chosen_tokenizer = tokenize_stop # kies je eigen tokenizer, bijvoorbeeld met of zonder stemming, met of zonder stopwoorden

tfidf = TfidfVectorizer(tokenizer=chosen_tokenizer, token_pattern=None, lowercase=False, min_df=1)
X_tfidf = tfidf.fit_transform(short_descriptions)
words = tfidf.get_feature_names_out()
print("matrix:", X_tfidf.shape)

print("Matrix (TF-IDF):\n", X_tfidf.toarray())

matrix: (283, 824)
Matrix (TF-IDF):
 [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [46]:
tfidf_dataframe = pd.DataFrame(X_tfidf.toarray(), columns=words, index=df["Titel"])

tfidf_dataframe.head(10)

,aandacht,aangrijpend,aangrijpende,aanleiding,aantrekkingskracht,aanwijzingen,academische,acceptatie,achtergrond,achtergronden,...,zoals,zoeken,zoekt,zoektocht,zomerse,zorg,zorgen,zwaar,zwangerschap,én
Titel,,,,,,,,,,,,,,,,,,,,,
het dagboek van anne frank,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0
blauw water,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0
boer vindt vrouw,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.222536,0.0,0.0,0.000000,0.0,0.0,0.0
cadeau voor u!,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0
de biecht,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0
de familie blues,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.206288,0.0,0.0,0.0
de favoriete plekken van...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0
de man van de bloemen,0.318659,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0
de ontmoeting,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0


In [47]:
for i in range(min(6, len(df))):
    # Rij i van de matrix: één getal per woord uit het vocabulaire.
    # .toarray() maakt er een gewone array van met vorm (1, aantal woorden), een "tabel" met één rij.
    # .ravel() maakt daar een platte lijst van (aantal woorden,), zodat je er met één index in kunt.
    row = X_tfidf[i].toarray().ravel()

    # row.argsort() geeft de POSITIES van de woorden, gesorteerd van laagste naar hoogste gewicht
    # (het geeft dus niet de gewichten zelf, maar waar ze staan).
    # [::-1] draait die volgorde om, zodat het hoogste gewicht vooraan staat.
    # [:3] pakt daarvan de eerste drie: de posities van de drie zwaarste woorden.
    # words[j] zoekt bij elke positie j het bijbehorende woord op.
    top = [words[j] for j in row.argsort()[::-1][:3]]
    print(f"{df['Titel'].iloc[i]:40} {top}")

het dagboek van anne frank               ['onderduikt', 'vervolging', 'joods']
blauw water                              ['terwijl', 'oploopt', 'beschermen']
boer vindt vrouw                         ['passende', 'plattelandsleven', 'decor']
cadeau voor u!                           ['verhalen', 'gewone', 'gebaren']
de biecht                                ['gang', 'betrokkenen', 'bekentenis']
de familie blues                         ['elkaar', 'relativerende', 'nodig']


In [48]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(X_tfidf)
print("gelijkenismatrix:", sim.shape)
pd.DataFrame(sim[:6, :6], index=df["Titel"].iloc[:6], columns=[t[:14] for t in df["Titel"].iloc[:6]]).round(2)

gelijkenismatrix: (283, 283)


,het dagboek va,blauw water,boer vindt vro,cadeau voor u!,de biecht,de familie blu
Titel,,,,,,
het dagboek van anne frank,1.00,0.00,0.00,0.00,0.03,0.00
blauw water,0.00,1.00,0.00,0.00,0.02,0.00
boer vindt vrouw,0.00,0.00,1.00,0.07,0.05,0.05
cadeau voor u!,0.00,0.00,0.07,1.00,0.05,0.08
de biecht,0.03,0.02,0.05,0.05,1.00,0.00
de familie blues,0.00,0.00,0.05,0.08,0.00,1.00


In [49]:
def top5(title: str, n: int = 5) -> pd.DataFrame:
    i = df["Titel"].tolist().index(title)
    scores = sim[i].copy()
    scores[i] = -1                                  # zichzelf overslaan
    best = scores.argsort()[::-1][:n]
    return pd.DataFrame({"titel": [df["Titel"].iloc[j] for j in best], "cosine": scores[best].round(3)})

title = df["Titel"].iloc[0]
print("Omdat je las:", title)
top5(title)

Omdat je las: het dagboek van anne frank


,titel,cosine
0,het meisje dat er niet mocht zijn,0.234
1,ik zal je vinden,0.192
2,de jongen in de gestreepte pyjama,0.189
3,neem de tijd,0.184
4,mooie mauti,0.133


In [50]:
title = "ik zal je vinden"
print("Omdat je las:", title)
top5(title)

Omdat je las: ik zal je vinden


,titel,cosine
0,het meisje dat er niet mocht zijn,0.560
1,boaz en de reis,0.375
2,terug naar aleppo,0.371
3,niemand zei iets,0.324
4,de foto van imke,0.302


Door naar het geven van gewichtenL

In [ ]:
from sklearn.preprocessing import normalize
from scipy.sparse import hstack

# Per kolom uit de rollentabel: welke vectorizer hoort erbij?
#   tekstkolom      -> TfidfVectorizer met tokenize_nostop
#   categoriekolom  -> CountVectorizer(binary=True): one-hot, meerdere labels per cel gescheiden door ";" mag
# Vul aan met de kolommen uit jullie eigen csv.
VECTORIZERS = {
    "Korte omschrijving": lambda: TfidfVectorizer(tokenizer=tokenize_stemmed_nostop, token_pattern=None, lowercase=False),
    "genre":       lambda: CountVectorizer(binary=True),
    "trefwoorden": lambda: CountVectorizer(binary=True),
}

weights = {"Korte omschrijving": 1.0, "genre": 0.3, }

def build_vectors_full(df, weights, return_columns=False, verbose=True):
    """weights: dict met één regel per blok, bijvoorbeeld
    {"Korte omschrijving": 1.0, "trefwoorden": 0.5, "genre": 0.3}

    Met return_columns=True komen ook de kolomnamen mee, als "<kolom>=<woord of label>"."""
    blocks, columns = [], []
    for col, make_vectorizer in VECTORIZERS.items():         # VECTORIZERS zegt wat er kán, weights wat er meedoet
        weight = weights.get(col, -1)                       # geen gewicht opgegeven = blok doet niet mee
        if weight == -1:
            if verbose: print(f"{col}: geen gewicht gezet voor kolom. Deze wordt op 1 gezet")
            weight = 1

        vectorizer = make_vectorizer()                       # () maakt een verse vectorizer uit de lambda
        text = df[col].fillna("").astype(str)                # lege cel -> "", anders weigert de vectorizer
        block = weight * normalize(vectorizer.fit_transform(text))
        blocks.append(block)
        names = vectorizer.get_feature_names_out()
        columns += [f"{col}={name}" for name in names]
        if verbose:
            print(f"{col}: {block.shape[1]} kolommen, gewicht {weight}, bv. {list(names[:6])}")
    X = hstack(blocks).tocsr()                               # alle blokken naast elkaar: (titels x features)
    return (X, columns) if return_columns else X

def sample_check(title_list, X, n=5):
    """Return a DataFrame with one row per (title, neighbour) and an empty 'oordeel' column
    to fill in by hand: goed / twijfel / fout."""
    rows = []
    for title in title_list:
        i = titles.index(title)
        sim = cosine_similarity(X[i], X).ravel()
        order = [j for j in np.argsort(-sim, kind="stable") if j != i][:n]
        for j in order:
            rows.append({"title": title, "neighbour": titles[j], "cosine": round(float(sim[j]), 2), "oordeel": ""})
    return pd.DataFrame(rows)

In [52]:
X, COLUMNS = build_vectors_full(df, weights, return_columns=True)
X.shape


Korte omschrijving: 745 kolommen, gewicht 1.0, bv. ['aandacht', 'aangrijp', 'aanleid', 'aantrekkingskracht', 'aanwijz', 'academisch']
genre: 3 kolommen, gewicht 0.3, bv. ['fictie', 'informatief', 'poëzie']
trefwoorden: geen gewicht gezet voor kolom. Deze wordt op 1 gezet
trefwoorden: 60 kolommen, gewicht 1, bv. ['bedrog', 'biografie', 'burgerschap', 'creativiteit', 'criminaliteit', 'cultuur']


(283, 808)

In [53]:
import importlib, functions
importlib.reload(functions)                 # functions.py is gewijzigd terwijl de kernel al liep
from functions import show_vectors

# Vijf titels uit de catalogus: informatief, twee keer fictie, één zonder genre en één dichtbundel.
demo = ["het dagboek van anne frank", "blauw water", "boer vindt vrouw", "de man van de bloemen"]
demo.append(df.loc[df["genre"] == "poëzie", "Titel"].iloc[0])
show_vectors(demo, X, COLUMNS, df, n_words=4)


,het dagboek van anne frank,blauw water,boer vindt vrouw,de man van de bloemen,jij bent de liefste
amsterdam,0.28,,,,
beeld,0.28,,,,
dagboek,0.28,,,,
jod,0.28,,,,
bescherm,,0.33,,,
oploopt,,0.33,,,
terwijl,,0.33,,,
bedreig,,0.3,,,
decor,,,0.35,,
passend,,,0.35,,


In [54]:

def similar_items(title, X, columns, books, n=5):
    """Top-n neighbours of a title on cosine similarity over the combined vector."""
    i = books["Titel"].tolist().index(title)
    sim = cosine_similarity(X[i], X).ravel()
    order = [j for j in np.argsort(-sim) if j != i][:n]
    return pd.DataFrame({"title": [books["Titel"].iloc[j] for j in order], "genre": books.loc[order, "genre"].values, "cosine": sim[order].round(2)})

similar_items("het dagboek van anne frank", X, COLUMNS, df)

,title,genre,cosine
0,floris,fictie,0.24
1,het meisje dat er niet mocht zijn,fictie,0.13
2,doe het toch maar,poëzie,0.13
3,neem de tijd,poëzie,0.12
4,sterren verpakt in huid,fictie,0.11


In [ ]:
def recommend(profile, n=10, exclude=()):
    """Top-n titles for a profile vector (one row, same columns as X)."""
    # exclude: posities (rijnummers) van titels die niet in de lijst mogen. Posities en geen titels,
    # want sommige titels staan meer dan eens in de catalogus. Hier komen ook de harde filters terecht.
    sim = cosine_similarity(profile, X).ravel()
    skip = set(exclude)
    order = [j for j in np.argsort(-sim, kind="stable") if j not in skip][:n]
    return pd.DataFrame({"cataloguscode": df["cataloguscode"].iloc[order].values,
                         "title": [titles[j] for j in order],
                         "genre": df["genre"].iloc[order].values,
                         "score": sim[order].round(2)})

for title in ["het dagboek van anne frank", "blauw water", "boer vindt vrouw"]:
    book = titles.index(title)
    profile = X[book]
    recommendations = recommend(profile, n=5, exclude=[book])
    print(f"Recommendations for '{title}':\n", recommendations, "\n")

## Profielen (uitwerking van 05 profile.ipynb)

De functies uit de les over profielen, uitgewerkt op de echte catalogus. Twee dingen zijn anders dan hierboven:

- `build_vectors` geeft ook de vectorizers terug (`VEC`). Die zijn nodig om een losse tekst door dezelfde TF-IDF te halen.
- Het trefwoordblok doet nu mee met een eigen gewicht; `X` wordt daarom opnieuw gebouwd.

In [ ]:
import re
from scipy.sparse import csr_matrix

WEIGHTS = {"Korte omschrijving": 1.0, "genre": 0.3, "trefwoorden": 0.3}

def build_vectors(df, weights):
    """Return (X, vectorizers): the same blocks as build_vectors_full, plus the fitted vectorizers per block."""
    vec, blocks = {}, []
    for col, make_vectorizer in VECTORIZERS.items():
        vec[col] = make_vectorizer()
        M = vec[col].fit_transform(df[col].fillna("").astype(str))
        blocks.append(weights.get(col, 0.0) * normalize(M.astype(float)))
    return hstack(blocks).tocsr(), vec

X, VEC = build_vectors(df, WEIGHTS)

# Trefwoorden als lijst per titel. De scheidingstekens in de catalogus zijn niet overal gelijk.
df["tag_list"] = df["trefwoorden"].fillna("").apply(
    lambda s: [t.strip().lower() for t in re.split(r"[;,|/]", s) if t.strip()])

print("X:", X.shape, "| blokken:", {col: len(v.vocabulary_) for col, v in VEC.items()})

In [ ]:
def build_profile(idx):
    """Mean of the rows of X at positions idx: one row, same columns as X."""
    return np.asarray(X[idx].mean(axis=0))

def profile_from_tags(tags):
    """A: mean vector of all titles that have at least one of the given tags."""
    wanted = {t.lower() for t in tags}
    idx = [i for i, tag_list in enumerate(df["tag_list"]) if wanted & set(tag_list)]
    return build_profile(idx), idx

def profile_from_categories(categories):
    """B: mean vector of all titles whose genre is in the list."""
    idx = df.index[df["genre"].isin([c.lower() for c in categories])].tolist()
    return build_profile(idx), idx

def profile_from_text(text, weights=WEIGHTS):
    """C: the same vectorizers as build_vectors, applied to one free text; genre and tag blocks stay empty."""
    blocks = []
    for col, v in VEC.items():                                   # zelfde volgorde als build_vectors
        if col == "Korte omschrijving":
            M = v.transform([text])                              # transform, niet fit_transform
        else:
            M = csr_matrix((1, len(v.vocabulary_)))              # leeg, maar even breed als het blok
        blocks.append(weights.get(col, 0.0) * normalize(M.astype(float)))
    return hstack(blocks).tocsr()

profile_A, idx_A = profile_from_tags(["vriendschap", "opgroeien"])
profile_B, idx_B = profile_from_categories(["poëzie"])
profile_C = profile_from_text("een spannend verhaal over een geheim uit de oorlog")
print("A kiest", len(idx_A), "titels, B kiest", len(idx_B), "titels")

pd.DataFrame({"A trefwoorden": recommend(profile_A, n=5)["title"].values,
              "B genres":      recommend(profile_B, n=5)["title"].values,
              "C eigen tekst": recommend(profile_C, n=5)["title"].values})

In [ ]:
# test: de omschrijving van een titel als zoektekst moet die titel bovenaan zetten
for t in ["blauw water", "het diner", "de voorlezer"]:
    profile_C = profile_from_text(df.loc[titles.index(t), "Korte omschrijving"])
    assert profile_C.shape == (1, X.shape[1]), ("zelfde kolommen als X", profile_C.shape, X.shape)
    top = recommend(profile_C, n=1)["title"][0]
    assert top == t, (t, top)
print("profile_from_text vindt de eigen omschrijving terug")

### De vragenlijst als dict: `person_profile`, `recommend_for` en `cold_start`

Keuzes in deze uitwerking:

- **Weging van bronnen.** Eigen woorden en trefwoorden tellen elk 0,4, genres 0,2: een genre is hier erg breed (twee derde van de catalogus is "fictie"). Ontbreekt een bron, dan worden de gewichten van de overige weer samen 1.
- **Eerst op lengte 1.** Een gemiddelde van honderd rijen is een veel kortere vector dan één tekst. Zonder normaliseren bepaalt de lengte van de vector hoe zwaar een bron telt, en niet het gewicht.
- **Harde filters.** `level` laat alleen titels op of onder dat niveau toe; titels zonder niveau vallen dan af. `avoid` haalt types materiaal weg.
- **Koude start.** Het vangnet: de hoogst beoordeelde titels die door de harde filters komen.

In [ ]:
SOURCE_WEIGHTS = {"tags": 0.4, "genres": 0.2, "free_text": 0.4}
LEVELS = ["2F", "3F", "3F+"]                 # van makkelijk naar moeilijk

def person_profile(person):
    """The profile vector for a person: a weighted mix of the sources that are filled in."""
    sources = {}
    if person.get("tags"):
        profile, idx = profile_from_tags(person["tags"])
        if idx:                                                  # onbekend trefwoord: geen titels, geen gemiddelde
            sources["tags"] = profile
    if person.get("genres"):
        profile, idx = profile_from_categories(person["genres"])
        if idx:
            sources["genres"] = profile
    if person.get("free_text"):
        profile = profile_from_text(person["free_text"])
        if profile.nnz:                                          # geen enkel bekend woord: lege vector
            sources["free_text"] = profile.toarray()
    if not sources:
        raise ValueError("geen enkele bron ingevuld: gebruik cold_start")

    total = sum(SOURCE_WEIGHTS[s] for s in sources)              # gewichten van de bronnen die er zijn: samen 1
    return sum(SOURCE_WEIGHTS[s] / total * normalize(p) for s, p in sources.items())

def excluded_positions(person):
    """Row positions that may never be recommended to this person: the hard filters."""
    keep = np.ones(len(df), dtype=bool)
    if person.get("level") in LEVELS:
        allowed = LEVELS[:LEVELS.index(person["level"]) + 1]
        keep &= df["Niveau"].isin(allowed).values
    if person.get("avoid"):
        keep &= ~df["Type materiaal"].isin(person["avoid"]).values
    return np.flatnonzero(~keep).tolist()

def cold_start(person, n=5):
    """Fallback for a reader who filled in nothing: the best rated titles that pass the hard filters."""
    skip = set(excluded_positions(person))
    ranked = df["Gemiddelde beoordeling"].sort_values(ascending=False, kind="stable").index      # NaN achteraan
    order = [j for j in ranked if j not in skip][:n]
    return pd.DataFrame({"cataloguscode": df["cataloguscode"].iloc[order].values,
                         "title": [titles[j] for j in order],
                         "genre": df["genre"].iloc[order].values,
                         "score": df["Gemiddelde beoordeling"].iloc[order].values})

def recommend_for(person, n=10):
    """Build a profile from whatever fields the person filled in, then recommend."""
    try:
        profile = person_profile(person)
    except ValueError:
        return cold_start(person, n=n)
    return recommend(profile, n=n, exclude=excluded_positions(person))

sam = {
    "name": "Sam",
    "tags": ["vriendschap", "opgroeien"],
    "genres": ["fictie"],
    "free_text": "een verhaal over vrienden die samen opgroeien en iets spannends meemaken",
    "level": "2F",
    "avoid": ["Dichtbundel"],
}
recommend_for(sam, n=10)

In [ ]:
# test: recommend_for werkt voor elke combinatie van velden, en voor een lezer die niets invulde
for fields in [["tags"], ["genres"], ["free_text"], ["tags", "genres"], ["tags", "genres", "free_text"]]:
    person = {k: sam[k] for k in fields}
    result = list(recommend_for(person, n=3)["title"])
    assert len(result) == 3, fields
    print(fields, "→", result)

for person in [{}, {"tags": [], "genres": [], "free_text": ""}]:
    result = list(recommend_for(person, n=5)["cataloguscode"])
    assert len(result) == 5 and len(set(result)) == 5, person
    print(person, "→", list(recommend_for(person, n=5)["title"]))

# harde filters: Sam (2F, geen dichtbundels) krijgt niets boven zijn niveau en geen dichtbundel
got = df.set_index("cataloguscode").loc[recommend_for(sam, n=20)["cataloguscode"]]
assert (got["Niveau"] == "2F").all() and (got["Type materiaal"] != "Dichtbundel").all()
print("harde filters kloppen")

## Evaluatie (uitwerking van 06 evaluation.ipynb)

Dezelfde zeven testlezers als in 06 evaluation.ipynb, de uitgewerkte maten, alle drie de ondergrenzen en `evaluate`. Onderaan staat de vergelijking: de drie ondergrenzen naast `recommend_for`.

In [ ]:
def codes(*numbers):
    """Catalogue codes from their numbers: codes(1, 17) -> ["CAT-00001", "CAT-00017"]."""
    return [f"CAT-{n:05d}" for n in numbers]

daan = {
    "name": "Daan",
    "tags": ["thriller", "mysterie", "geheimen"],
    "genres": ["fictie"],
    "free_text": "een spannende thriller of detective met geheimen en een onverwachte ontknoping",
    "level": "2F",
    "avoid": [],
    "relevant": codes(54, 57, 66, 87, 89, 94, 98, 101, 102, 103, 108, 109, 110, 114, 121, 123, 124, 125,
                      133, 134, 135, 191, 202, 207, 240, 247, 253, 266, 277),
}

thomas = {
    "name": "Thomas",
    "tags": ["oorlog", "WOII", "onderduik"],
    "genres": ["fictie"],
    "free_text": "waargebeurde verhalen uit de Tweede Wereldoorlog, over onderduiken en overleven",
    "level": "3F",
    "avoid": [],
    "relevant": codes(1, 10, 11, 17, 32, 33, 42, 46, 85, 248),
}

fatima = {
    "name": "Fatima",
    "tags": ["migratie", "hoop"],
    "genres": ["fictie"],
    "free_text": "verhalen over mensen die naar een nieuw land komen en daar opnieuw moeten beginnen",
    "level": "2F",
    "avoid": [],
    "relevant": codes(22, 43, 77, 188, 206, 217, 222, 223, 271, 277),
}

milan = {
    "name": "Milan",
    "tags": ["sport"],
    "genres": [],
    "free_text": "boeken over sport, en ik wil ook weten wat AI voor mij gaat betekenen",
    "level": "3F",
    "avoid": [],
    "relevant": codes(48, 106, 111, 115, 184, 240, 268, 272,                    # sport
                      139, 140, 148, 152, 153, 175, 178, 183, 209, 280),        # AI
}

lotte = {
    "name": "Lotte",
    "tags": [],
    "genres": [],
    "free_text": "ik zoek iets over rouw: iemand die een ouder of een vriend verliest en daarmee moet leren leven",
    "level": "3F",
    "avoid": [],
    "relevant": codes(21, 79, 117, 136, 198, 201, 205, 213, 219, 234, 276),
}

youssef = {
    "name": "Youssef",
    "tags": ["criminaliteit", "recht"],
    "genres": [],
    "free_text": "hoe werkt de rechtspraak, en wat gebeurt er met jongeren die de fout in gaan",
    "level": "3F",
    "avoid": ["Online artikel", "Blogpost"],
    "relevant": codes(50, 64, 69, 71, 73, 74, 76, 78, 87, 91, 93),
}

emma = {
    "name": "Emma",
    "tags": [],
    "genres": ["poëzie"],
    "free_text": "gedichten over identiteit en opgroeien",
    "level": "2F",
    "avoid": [],
    "relevant": codes(160, 161, 252),
}

fixed_readers = [daan, thomas, fatima, milan, lotte, youssef, emma]      # liggen vast: niet aanpassen

# Jullie eigen drie testlezers. Zelfde velden als hierboven; "relevant" is een lijst cataloguscodes.
# Voorbeeld:
# reader_8 = {
#     "name": "...",
#     "tags": [...],
#     "genres": [...],
#     "free_text": "...",
#     "level": "2F",              # 2F, 3F of 3F+
#     "avoid": [],                # types materiaal die deze lezer niet wil
#     "relevant": codes(...),     # titels die deze lezer zou moeten krijgen
# }
# Titel en omschrijving bij een lijst codes: catalog.loc[codes(1, 17), ["Titel", "Niveau", "Korte omschrijving"]]
own_readers = []

readers = fixed_readers + own_readers

pd.DataFrame({"name": [r["name"] for r in readers],
              "level": [r["level"] for r in readers],
              "ingevuld": [", ".join(f for f in ["tags", "genres", "free_text"] if r[f]) for r in readers],
              "aantal relevant": [len(r["relevant"]) for r in readers]})

In [ ]:
def precision_at_k(recommended, relevant, k):
    """Share of the top-k recommended titles that are relevant."""
    hits = sum(1 for t in recommended[:k] if t in set(relevant))
    return hits / k

def recall_at_k(recommended, relevant, k):
    """Share of the relevant titles that appear in the top-k."""
    if not relevant:
        return 0.0
    hits = sum(1 for t in recommended[:k] if t in set(relevant))
    return hits / len(relevant)

def hit_at_k(recommended, relevant, k):
    """1 when at least one relevant title is in the top-k, otherwise 0."""
    return 1 if any(t in set(relevant) for t in recommended[:k]) else 0

def evaluate(recommender, readers, k=5):
    """One row per reader with precision, recall and hit at k."""
    rows = []
    for reader in readers:
        person = {key: value for key, value in reader.items() if key != "relevant"}     # de verwachting blijft hier
        recommended = recommender(person)
        rows.append({"name": reader["name"],
                     "precision": precision_at_k(recommended, reader["relevant"], k),
                     "recall": recall_at_k(recommended, reader["relevant"], k),
                     "hit": hit_at_k(recommended, reader["relevant"], k)})
    return pd.DataFrame(rows, columns=["name", "precision", "recall", "hit"])

In [ ]:
# test: letters in plaats van codes, zodat je het met de hand kunt narekenen
rec = ["a", "b", "c", "d", "e"]
rel = ["b", "e", "z"]

assert precision_at_k(rec, rel, k=3) == 1 / 3
assert recall_at_k(rec, rel, k=3) == 1 / 3          # 1 van de 3 relevante titels
assert recall_at_k(rec, rel, k=5) == 2 / 3          # "z" wordt nooit aanbevolen
assert recall_at_k(rec, [], k=5) == 0.0             # geen verwachting: geen recall
assert hit_at_k(rec, rel, k=1) == 0                 # "a" is geen treffer
assert hit_at_k(rec, rel, k=2) == 1
assert hit_at_k([], rel, k=5) == 0                  # lege lijst: geen treffer
print("recall_at_k en hit_at_k kloppen")

In [ ]:
# test: een recommender met een vaste lijst voor Emma en Lotte, met de hand na te rekenen
FIXED = {
    "Emma":  codes(160, 1, 161, 2, 3),       # 160 en 161 zijn relevant voor Emma
    "Lotte": codes(5, 21, 6, 7, 8),          # alleen 21 is relevant voor Lotte
}

def fixed_recommender(person):
    assert "relevant" not in person, "de recommender krijgt het veld relevant te zien"
    return FIXED[person["name"]]

result = evaluate(fixed_recommender, [emma, lotte], k=3)
assert list(result.columns) == ["name", "precision", "recall", "hit"], list(result.columns)
assert list(result["name"]) == ["Emma", "Lotte"]

by_name = result.set_index("name")
assert abs(by_name.loc["Emma", "precision"] - 2 / 3) < 1e-9                              # 2 treffers in de top 3
assert abs(by_name.loc["Emma", "recall"] - 2 / len(emma["relevant"])) < 1e-9
assert abs(by_name.loc["Lotte", "precision"] - 1 / 3) < 1e-9
assert abs(by_name.loc["Lotte", "recall"] - 1 / len(lotte["relevant"])) < 1e-9
assert list(result["hit"]) == [1, 1]
assert list(evaluate(fixed_recommender, [emma, lotte], k=1)["hit"]) == [1, 0]

print("evaluate klopt")
result

In [ ]:
import random

catalog = df.set_index("cataloguscode")

def random_baseline(person, n=10):
    """Baseline 1: n random titles. The same reader always gets the same list."""
    rng = random.Random(person.get("name", ""))
    return rng.sample(list(catalog.index), n)

def popular_baseline(person, n=10):
    """Baseline 2: the n best rated titles, the same list for everybody."""
    return list(catalog["Gemiddelde beoordeling"].sort_values(ascending=False, kind="stable").index[:n])

def genre_baseline(person, n=10):
    """Baseline 3: random titles from the genres the reader ticked; no genre ticked means the whole catalogue."""
    pool = list(catalog.index[catalog["genre"].isin(person.get("genres") or [])]) or list(catalog.index)
    rng = random.Random(person.get("name", ""))
    return rng.sample(pool, min(n, len(pool)))

baseline_recommender = genre_baseline           # de gekozen ondergrens; de andere twee staan in de tabel hieronder

def my_recommender(person):
    """Our own recommender in the shape the evaluation expects: a list of codes, best first."""
    return list(recommend_for(person, n=10)["cataloguscode"])

variants = {"willekeurig": random_baseline, "populairst": popular_baseline,
            "zelfde genre": genre_baseline, "recommend_for": my_recommender}
pd.DataFrame({name: evaluate(rec, readers, k=5)[["precision", "recall", "hit"]].mean()
              for name, rec in variants.items()}).T.round(2)

In [ ]:
# per lezer: voor wie gaat het goed, voor wie niet?
ours = evaluate(my_recommender, readers, k=5)
ours["top 5"] = [", ".join(catalog.loc[my_recommender(r)[:5], "Titel"]) for r in readers]
pd.set_option("display.max_colwidth", 200)
ours.round(2)

## Uitleg (uitwerking van 07 explainability.ipynb)

Waarom staat een titel in de lijst? De cosine tussen een profiel en een titel is een som over kolommen, en elke kolom is een woord of een label. De bijdrage van kolom *j* is `profiel[j] * titel[j]`, gedeeld door de lengtes van beide rijen. De bijdragen tellen op tot de cosine, dus de uitleg is **waar**: hij komt uit dezelfde berekening als de score.

Keuzes in deze uitwerking:

- **Kolomnamen komen uit `VEC`.** `X` is de blokken uit `VECTORIZERS` achter elkaar, dus de namen moeten in dezelfde volgorde. Een naam heeft de vorm `<kolom>=<woord>`, zoals in `build_vectors_full`. Een assert op `len(NAMES) == X.shape[1]` vangt een verschoven blok af.
- **Blokgrenzen uit de breedtes** van de vectorizers (`EDGES`), niet uit het tellen van namen.
- **Een titel op positie.** Sommige titels staan meer dan eens in de catalogus, dus `explain` werkt met de cataloguscode en `POS` zoekt de rij in `X`.
- **Het profiel is wat `person_profile` geeft**, dus ook de gemengde bronnen (trefwoorden, genres, eigen woorden) zijn uit te leggen.

In [ ]:
POS = {code: i for i, code in enumerate(df["cataloguscode"])}        # cataloguscode -> rijnummer in X

def feature_names(vec):
    """Names of all columns of X: '<kolom>=<woord>', blocks in the order of VECTORIZERS (= the order in X)."""
    names = []
    for col in VECTORIZERS:
        names += [f"{col}={w}" for w in vec[col].get_feature_names_out()]
    return names

NAMES = feature_names(VEC)
assert len(NAMES) == X.shape[1], (len(NAMES), X.shape[1])
assert len(set(NAMES)) == len(NAMES), "namen moeten uniek zijn"

WIDTHS = [len(VEC[col].get_feature_names_out()) for col in VECTORIZERS]
EDGES = np.cumsum([0] + WIDTHS)                              # blok i = kolommen EDGES[i] t/m EDGES[i+1]
assert EDGES[-1] == X.shape[1]
print(dict(zip(VECTORIZERS, WIDTHS)), "grenzen:", EDGES.tolist())

In [ ]:
def contributions(profile, code):
    """Per column: how much does it add to the cosine between the profile and the title with this code?"""
    p_row = np.asarray(profile.todense() if hasattr(profile, "todense") else profile, dtype=float).ravel()
    x_row = X[POS[code]].toarray().ravel()
    norm = np.linalg.norm(p_row) * np.linalg.norm(x_row)
    if norm == 0:                                            # leeg profiel of lege titelrij
        return np.zeros(X.shape[1])
    return p_row * x_row / norm

def explain(profile, code, k=3):
    """The k columns (words or labels) that contribute most to the cosine; only columns with a contribution > 0."""
    c = contributions(profile, code)
    top = [j for j in np.argsort(-c, kind="stable") if c[j] > 0][:k]
    return [(NAMES[j], round(float(c[j]), 3)) for j in top]

def explain_by_block(profile, code):
    """The share of each block (Korte omschrijving, genre, trefwoorden) in the cosine; the shares add up to 1."""
    c = contributions(profile, code)
    total = c.sum()
    if total == 0:
        return {col: 0.0 for col in VECTORIZERS}
    return {col: float(c[EDGES[i]:EDGES[i + 1]].sum() / total) for i, col in enumerate(VECTORIZERS)}

In [ ]:
# test: de bijdragen tellen op tot de cosine, en elk genoemd woord of label komt echt in de titel voor
def occurs(name, code):
    col, word = name.split("=", 1)
    row = df.iloc[POS[code]]
    analyzer = VEC[col].build_analyzer()
    return word in analyzer(str(row[col]) if pd.notna(row[col]) else "")

for person in readers:
    profile = person_profile(person)
    for code in recommend_for(person, n=3)["cataloguscode"]:
        c = contributions(profile, code)
        assert (c >= 0).all()
        assert abs(c.sum() - cosine_similarity(profile, X[POS[code]])[0, 0]) < 1e-9, (person["name"], code)
        result = explain(profile, code, k=3)
        assert 1 <= len(result) <= 3, (person["name"], code, result)
        values = [v for _, v in result]
        assert values == sorted(values, reverse=True) and all(v > 0 for v in values)
        assert all(occurs(name, code) for name, _ in result), (person["name"], code, result)
        assert abs(sum(explain_by_block(profile, code).values()) - 1) < 1e-9
print("contributions, explain en explain_by_block kloppen")

In [ ]:
# de nummer 1 van elke testlezer, met de woorden en het blokaandeel
for person in readers:
    profile = person_profile(person)
    code_1 = recommend_for(person, n=1)["cataloguscode"].iloc[0]
    share = {col: round(v, 2) for col, v in explain_by_block(profile, code_1).items()}
    print(f"{person['name']:8s} -> {df.iloc[POS[code_1]]['Titel']}")
    print(f"           woorden: {explain(profile, code_1, k=3)}")
    print(f"           blokken: {share}")

### Van getallen naar een zin: `explain_text`

Een lezer ziet geen `Korte omschrijving=oorlog (0,13)`. De keuze in deze uitwerking: **woorden uit de omschrijving en trefwoorden noemen, het genre apart, en niets zeggen als er geen duidelijk woord is**. Een uitleg blijft waar: er staat alleen wat in de bijdragen zit. Bij een titel die vooral op genre scoort noemt de zin dat ook zo, in plaats van een verhaal te verzinnen over wat de lezer 'leuk vindt'.

In [ ]:
def explain_text(person, code, k=3):
    """One sentence for the reader on why this title is in the list."""
    words = {col: [] for col in VECTORIZERS}
    for name, _ in explain(person_profile(person), code, k):
        col, word = name.split("=", 1)
        words[col].append(word)
    pieces = []
    if words["Korte omschrijving"]:
        pieces.append("de omschrijving gaat over " + ", ".join(f"'{w}'" for w in words["Korte omschrijving"]))
    if words["trefwoorden"]:
        pieces.append("de trefwoorden zijn " + ", ".join(f"'{w}'" for w in words["trefwoorden"]))
    if words["genre"]:
        pieces.append("het genre is " + ", ".join(f"'{w}'" for w in words["genre"]))
    if not pieces:
        return "Deze titel past bij je profiel, maar ik kan geen duidelijk woord aanwijzen."
    return "Past bij je omdat " + "; ".join(pieces) + "."

for person in readers:
    code_1 = recommend_for(person, n=1)["cataloguscode"].iloc[0]
    sentence = explain_text(person, code_1)
    top_word = explain(person_profile(person), code_1, k=1)[0][0].split("=", 1)[1]
    assert top_word in sentence, (person["name"], top_word, sentence)
for person in [thomas, lotte, milan]:
    code_1 = recommend_for(person, n=1)["cataloguscode"].iloc[0]
    print(f"{person['name']:8s} {df.iloc[POS[code_1]]['Titel']}: {explain_text(person, code_1)}")

### Uitleg als debughulpmiddel: waarom ontbreekt deze titel?

Bij een lage recall staat een relevante titel niet in de top 5. `explain` laat zien wat hij wél deelt met het profiel, en `missing_words` welke zware kolommen van het profiel de titel mist. Dat is meestal een catalogus-kwestie (een trefwoord dat ontbreekt) of een recommender-kwestie (een woord dat niet door de tokenizer als hetzelfde woord wordt gezien).

In [ ]:
def top_profile_words(profile, k=8):
    """The k heaviest columns of a profile, with their weight."""
    row = np.asarray(profile.todense() if hasattr(profile, "todense") else profile, dtype=float).ravel()
    return [(NAMES[j], round(float(row[j]), 3)) for j in np.argsort(-row, kind="stable")[:k] if row[j] > 0]

def missing_words(profile, code, k=8):
    """Heavy profile columns that the title does not have: why does this title score lower?"""
    row = X[POS[code]].toarray().ravel()
    heavy = top_profile_words(profile, 3 * k)
    return [(name, w) for name, w in heavy if row[NAMES.index(name)] == 0][:k]

person = thomas
profile = person_profile(person)
top5 = list(recommend_for(person, n=5)["cataloguscode"])
missed = [c for c in person["relevant"] if c not in top5]
sim_all = dict(zip(df["cataloguscode"], cosine_similarity(profile, X).ravel()))
code_missed = max(missed, key=sim_all.get)                    # de gemiste titel die het dichtst bij kwam
print(f"{person['name']}: {len(missed)} van {len(person['relevant'])} relevante titels niet in de top 5\n")
print("gemist:  ", df.iloc[POS[code_missed]]["Titel"], "| cosine", round(sim_all[code_missed], 3))
print("  wel gemeen:   ", explain(profile, code_missed, k=5))
print("  profiel mist: ", missing_words(profile, code_missed, k=5))
print("nummer 5:", df.iloc[POS[top5[-1]]]["Titel"], "| cosine", round(sim_all[top5[-1]], 3))
print("  wel gemeen:   ", explain(profile, top5[-1], k=5))